In [7]:
import pandas as pd
import numpy as np
import base64
import os
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

In [8]:
# 1. 초기 설정
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
df = pd.read_csv('res/res_TV.csv')
texts = df['content'].fillna("").tolist()

In [4]:
# 2. 빈 값 및 결측치 전처리
print(f"원본 데이터 개수: {len(df)}개")

# (1) content 컬럼이 아예 비어있는(NaN) 행 제거
df = df.dropna(subset=['content'])

# (2) 공백만 있거나(" "), 글자 수가 0인 행 제거
df = df[df['content'].astype(str).str.strip().str.len() > 0]

# (3) 인덱스 초기화
df = df.reset_index(drop=True)


texts = df['content'].tolist()
print(f"전처리 후 처리할 데이터 개수: {len(df)}개")

원본 데이터 개수: 28883개
전처리 후 처리할 데이터 개수: 28858개


In [5]:
# 3. 임베딩 및 블롭 변환 함수
def get_blob_embeddings(text_list, batch_size=2000):
    all_blobs = []
    
    for i in tqdm(range(0, len(text_list), batch_size), desc="변환 중"):
        batch = text_list[i:i+batch_size]
        response = client.embeddings.create(
            input=batch,
            model="text-embedding-3-small"
        )
        
        for data in response.data:
            # 숫자를 float32 배열로 변환 후 바이너리로 직렬화 -> Base64 인코딩
            vec = np.array(data.embedding, dtype=np.float32)
            blob = base64.b64encode(vec.tobytes()).decode('utf-8')
            all_blobs.append(blob)
            
    return all_blobs

In [ ]:
# 4. 실행 및 저장
df['embedding_blob'] = get_blob_embeddings(texts)

# 파인콘 업로드용 CSV 저장 (인덱스 포함)
df.to_csv('TV_embeddings.csv', index=False, encoding='utf-8-sig')

print("CSV가 저장되었습니다.")

변환 중: 100%|██████████| 15/15 [01:08<00:00,  4.54s/it]


CSV가 저장되었습니다.
